In [24]:
import numpy as np
import pandas as pd

In [25]:
def load_santa_fe(path):
    """
    Load the Santa Fe competition datasets.
    """
    
    with open(path) as f:
        text = f.read()
    values = [v for v in text.replace("\n", ",").split(",") if v.strip()]
    return pd.DataFrame(values, columns=["intensity"])

In [26]:
# Load datasets.
sunspots = pd.read_csv("./Datasets/SN_m_tot_V2.0.csv", header=None, names=["year", "month", "decimal_date", "sunspot_number", "std_dev", "n_obs", "definitive"])
flights = pd.read_csv("./Datasets/airline-passengers.csv")
temperatures = pd.read_csv("./Datasets/daily-min-temperatures.csv", index_col=0, parse_dates=True)
mackey_glass = pd.read_csv("./Datasets/Mackey-Glass Time Series (taw17).csv", index_col=0)
santa_fe_train = load_santa_fe("./Datasets/Santa Fe/SF_A.dat")
santa_fe_test = load_santa_fe("./Datasets/Santa Fe/SF_Acont.dat")

# Data pre-processing

## Mackey-Glass data

In [27]:
def swap_columns(df, col1, col2):
    col_list = list(df.columns)
    x, y = col_list.index(col1), col_list.index(col2)
    col_list[y], col_list[x] = col_list[x], col_list[y]
    df = df[col_list]
    return df

In [28]:
# Swap t and t-tau columns and rename both.
mackey_glass_swapped = swap_columns(mackey_glass, "t", "t-tau")
mackey_glass_swapped.rename(columns={"t-tau": "t", "t": "t-tau", "t+1": "next_t"}, inplace=True)

In [29]:
# Recreate the dataset from only the t-column.
t = mackey_glass_swapped["t"]
mackey_glass_final = pd.DataFrame({
    "t": t,
    "t-tau": t.shift(17),    # t-value 17 steps ago -> NaN in first 17 rows
    "t_next": t.shift(-1),   # next t-value -> NaN in last row
})
mackey_glass_final = mackey_glass_final.dropna()             # keep only rows where every column is real

In [30]:
# Remove the first and last 17 rows since they contain a "0.0" value for t or t-tau due to padding.
mackey_glass_final = mackey_glass_final[1:-17]

In [43]:
mackey_glass_final.to_csv("mackey_glass_final.csv", index=False)

## Melbourne temperatures dataset

In [31]:
temperatures

,Temp
Date,
1981-01-01,20.7
1981-01-02,17.9
1981-01-03,18.8
1981-01-04,14.6
1981-01-05,15.8
...,...
1990-12-27,14.0
1990-12-28,13.6
1990-12-29,13.5


In [32]:
# There are 2 missing dates in the dataset. We need to add them as rows with NaN values.
full_date_range = pd.date_range(temperatures.index.min(), temperatures.index.max(), freq="D")
temperatures = temperatures.reindex(full_date_range) # the two gaps now show up as NaN rows

In [38]:
temperatures["Temp"] = temperatures["Temp"].ffill()

In [44]:
temperatures.to_csv("./Pre-processed datasets/temperatures.csv")

## Airline dataset